In [1]:
!pip install -q wrds rapidfuzz pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import wrds
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

db = wrds.Connection()  # caches credentials in ~/.pgpass after first login

OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: fe_sendauth: no password supplied

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# ── Pull the 18 raw Compustat line items matching the sowide dataset paper ──
# act=current_assets  at=total_assets  cogs=COGS  dltt=LT_debt
# dp=D&A  ebit=EBIT  oibdp=EBITDA  gp=gross_profit  invt=inventory
# lct=current_liabilities  ni=net_income  re=retained_earnings
# rect=receivables  revt=total_revenue  lt=total_liabilities
# sale=net_sales  xopr=total_opex  (mktval computed below)

fin = db.raw_sql("""
    SELECT gvkey, cik, tic, conm, sic, fyear, datadate,
           act, at, cogs, dltt, dp, ebit, oibdp, gp, invt,
           lct, ni, re, rect, revt, lt, sale, xopr,
           csho, prcc_f
    FROM comp.funda
    WHERE indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C'
      AND exchg IN (11, 14)
      AND fyear BETWEEN 1999 AND 2018
      AND at > 0
""", date_cols=["datadate"])

fin["mktval"] = fin["csho"] * fin["prcc_f"]
fin["gp"]    = fin["gp"].fillna(fin["sale"] - fin["cogs"])      # fallback
fin["xopr"]  = fin["xopr"].fillna(fin["sale"] - fin["ebit"])     # fallback
fin = fin.drop(columns=["csho", "prcc_f"]).sort_values(["gvkey", "fyear"]).reset_index(drop=True)

print(fin.shape, "|", fin["gvkey"].nunique(), "companies")

In [ ]:
# ── Bankruptcy labels: last observed fyear for Chapter 7/11 filers = 'failed' ──
bk = db.raw_sql("""
    SELECT gvkey FROM comp.company WHERE dlrsn = '02'
""")

last_fyear = fin.groupby("gvkey")["fyear"].max()
failed_set = set(zip(
    bk["gvkey"],
    bk["gvkey"].map(last_fyear)
))

fin["status_label"] = fin.apply(
    lambda r: "failed" if (r["gvkey"], r["fyear"]) in failed_set else "alive", axis=1
)
print(fin["status_label"].value_counts())

In [ ]:
# ── Build 8-year rolling lag windows per company (mirrors the LSTM input shape) ──
FEAT_COLS = ["act","at","cogs","dltt","dp","ebit","oibdp","gp",
             "invt","lct","ni","re","rect","revt","mktval","lt","sale","xopr"]

def attach_lags(grp):
    parts = [grp]
    for lag in range(1, 8):
        shifted = grp[FEAT_COLS].shift(lag).add_prefix(f"lag{lag}_")
        parts.append(shifted)
    return pd.concat(parts, axis=1)

fin_lagged = (
    fin.groupby("gvkey", group_keys=False)
       .apply(attach_lags)
       .dropna(subset=[f"lag7_{FEAT_COLS[0]}"])   # require full 8-year window
       .reset_index(drop=True)
)
print(fin_lagged.shape)

In [ ]:
# ── Aggregate Glassdoor reviews → one row per (firm, fyear) ──
gd = pd.read_csv("glassdoor_reviews.csv", low_memory=False, parse_dates=["date_review"])
gd["fyear"] = gd["date_review"].dt.year

RATING_COLS = ["overall_rating","work_life_balance","culture_values",
               "career_opp","comp_benefits","senior_mgmt"]

agg = (
    gd.groupby(["firm", "fyear"])
      .agg(
          **{c: (c, "mean") for c in RATING_COLS},
          review_count=("overall_rating", "count"),
          # concatenate all text for FinBERT encoding in later notebooks
          gd_text=("pros", lambda x:
              " ".join(gd.loc[x.index, ["headline","pros","cons"]]
                          .fillna("").apply(" ".join, axis=1)))
      )
      .reset_index()
)
print(agg.shape)

In [ ]:
# ── Fuzzy-match Glassdoor firm names → Compustat gvkey ──
OVERRIDES = {
    "J-P-Morgan":             "JPMORGAN CHASE & CO",
    "Goldman-Sachs":          "GOLDMAN SACHS GROUP INC",
    "BNY-Mellon":             "BANK OF NEW YORK MELLON CORP",
    "Thomson-Reuters":        "THOMSON REUTERS CORP",
    "Facebook":               "META PLATFORMS INC",
    "Google":                 "ALPHABET INC",
    "Microsoft":              "MICROSOFT CORP",
    "Apple":                  "APPLE INC",
    "IBM":                    "INTERNATIONAL BUSINESS MACHINES CORP",
    "Oracle":                 "ORACLE CORP",
    "Cisco-Systems":          "CISCO SYSTEMS INC",
    "Morgan-Stanley":         "MORGAN STANLEY",
    "Citi":                   "CITIGROUP INC",
    "American-Express":       "AMERICAN EXPRESS CO",
    "Mastercard":             "MASTERCARD INC",
    "McDonald-s":             "MCDONALDS CORP",
    "Marriott-International": "MARRIOTT INTERNATIONAL INC",
    "Hilton":                 "HILTON WORLDWIDE HOLDINGS INC",
    "Hyatt":                  "HYATT HOTELS CORP",
    "Salesforce":             "SALESFORCE INC",
    "VMware":                 "VMWARE INC",
    "Workday":                "WORKDAY INC",
    "SAP":                    "SAP SE",
    "AstraZeneca":            "ASTRAZENECA PLC",
    "GlaxoSmithKline":        "GLAXOSMITHKLINE PLC",
    "Unilever":               "UNILEVER PLC",
    # private / no Compustat entry → skip
    "Bloomberg-L-P":          None,
    "LinkedIn":               None,
    "Indeed":                 None,
    "McKinsey-and-Company":   None,
}

name_to_gvkey = (
    fin_lagged[["gvkey", "conm"]].drop_duplicates()
                                  .set_index("conm")["gvkey"]
                                  .to_dict()
)
comp_names   = list(name_to_gvkey)
comp_cleaned = [n.upper().replace(" INC","").replace(" CORP","")
                 .replace(" LTD","").replace(" PLC","").strip()
                for n in comp_names]

def resolve(gd_firm):
    if gd_firm in OVERRIDES:
        target = OVERRIDES[gd_firm]
        return name_to_gvkey.get(target) if target else None
    query = gd_firm.replace("-", " ").upper()
    match, score, idx = process.extractOne(query, comp_cleaned, scorer=fuzz.token_sort_ratio)
    return name_to_gvkey[comp_names[idx]] if score >= 80 else None

firm_to_gvkey = {f: resolve(f) for f in agg["firm"].unique()}
agg["gvkey"]  = agg["firm"].map(firm_to_gvkey)

matched = agg["gvkey"].notna()
print(f"Matched {matched.sum()}/{len(agg)} company-year rows  "
      f"({agg.loc[matched,'firm'].nunique()} of {agg['firm'].nunique()} firms)")

# Inspect low-confidence or unmatched firms
unmatched = [f for f, g in firm_to_gvkey.items() if g is None]
print("Unmatched:", unmatched)

In [ ]:
# ── Final join → master dataset ──
master = (
    fin_lagged
    .merge(
        agg.dropna(subset=["gvkey"]).drop(columns=["firm"]),
        on=["gvkey", "fyear"],
        how="left"
    )
    .rename(columns={"conm": "company_name"})
)

print(master.shape)
print(master["status_label"].value_counts())
print(f"Rows with Glassdoor data: {master['review_count'].notna().sum()}")

master.to_parquet("master_dataset.parquet", index=False)
print("Saved → master_dataset.parquet")